# 01b — Transformer les variables : comment et pourquoi ?

Ce notebook montre **avant/après** comment les colonnes de `01_baseline.xlsx` deviennent des entrées numériques pour les modèles. À lire après l’exploration et avant les notebooks 04/05.

On réutilise leurs prétraitements de `src/modeling.py`. Aucun modèle n’est entraîné ici ; aucune nouvelle copie Excel ou CSV encodée n’est créée. Encoder une variable ne corrige pas sa qualité et ne démontre pas son utilité prédictive.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "modeling.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "modeling.py").exists():
    raise RuntimeError("Lancer depuis la racine ou notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from hashlib import sha256
import numpy as np
import pandas as pd
from IPython.display import display
from src.modeling import (DATA_PATH, BINARY, NOMINAL, SCORES, EXCLUDED,
    load_model_data, make_split, build_pipelines, encode_binary)
source_hash = sha256(DATA_PATH.read_bytes()).hexdigest()
X, y, source_rows, groups, quantitative = load_model_data()
X_before = X.copy(deep=True)
X_train, X_test, y_train, y_test, folds, split_rows = make_split(X, y, source_rows)
print(DATA_PATH.relative_to(ROOT))
print("Entraînement :", X_train.shape, "— test :", X_test.shape)
display(groups)

data/variantes/01_baseline.xlsx
Entraînement : (1214, 30) — test : (304, 30)


,variable,groupe
0,Gender,binaire
1,OverTime,binaire
2,BusinessTravel,nominale
3,Department,nominale
4,EducationField,nominale
5,JobRole,nominale
6,MaritalStatus,nominale
7,Education,score_ordonne
8,EnvironmentSatisfaction,score_ordonne
9,JobInvolvement,score_ordonne


## 1. Séparer cible et variables explicatives
`Attrition` est la réponse à prévoir : No=0, Yes=1. La colonne dérivée `depart` porte la même information : elle doit aussi être exclue des entrées. Sinon le modèle aurait déjà la réponse, ce qui serait une fuite de cible.

L’identifiant ne décrit pas un facteur RH. Les trois constantes et l’identifiant ont déjà été retirés de la baseline. Le numéro de ligne Excel est conservé à part, pour retrouver une observation, sans le donner au modèle.

In [2]:
display(pd.DataFrame({"ligne_excel": source_rows.head(6),
    "Attrition_avant": y.head(6).map({0: "No", 1: "Yes"}), "cible_apres": y.head(6)}))
assert not set(EXCLUDED + ["ligne_excel"]).intersection(X.columns)
print("La cible n’apparaît pas dans X.")

,ligne_excel,Attrition_avant,cible_apres
0,2,Yes,1
1,3,No,0
2,4,Yes,1
3,5,No,0
4,6,No,0
5,7,No,0


La cible n’apparaît pas dans X.


## 2. Encoder les variables binaires en 0/1
Deux états peuvent être représentés par une indicatrice. Gender : Male=0, Female=1 ; OverTime : No=0, Yes=1. Le codage est une convention technique, pas un ordre de valeur entre personnes.

Le mapping est fixe : il n’apprend rien des autres salariés. Un code inattendu déclenche une erreur explicite plutôt qu’une conversion silencieuse en valeur manquante. Ces colonnes ne sont pas standardisées dans nos modèles.

In [3]:
binary_before = X_train[list(BINARY)].head(8)
binary_after = encode_binary(binary_before)
display(pd.concat({"avant": binary_before, "apres": binary_after}, axis=1))
print(BINARY)

avant           apres         
      Gender OverTime Gender OverTime
1466    Male       No      0        0
964   Female       No      1        0
975     Male      Yes      0        1
1280    Male       No      0        0
662   Female       No      1        0
541   Female       No      1        0
839     Male       No      0        0
613     Male       No      0        0

{'Gender': {'Male': 0, 'Female': 1}, 'OverTime': {'No': 0, 'Yes': 1}}


## 3. Pourquoi le one-hot encoding pour les catégories nominales ?
Un métier ou un département n’a pas d’ordre numérique naturel. Coder Sales=1, Research & Development=2 et Human Resources=3 introduirait des distances arbitraires dans une régression.

Le **one-hot encoding** crée une colonne par modalité : 1 si le salarié appartient à cette catégorie, 0 sinon. Un salarié n’appartient qu’à une modalité par variable nominale. Exemple ci-dessous : les catégories réellement présentes de BusinessTravel ; aucun libellé n’est inventé.

Nous conservons toutes les modalités (`drop=None`) pour cette régression prédictive L2. Une régression non pénalisée avec constante nécessiterait de traiter la dépendance entre indicatrices, par exemple en retirant une référence. Ici, les coefficients isolés ne doivent pas se lire comme des écarts à une référence supprimée.

In [4]:
# On ajuste les prétraitements existants uniquement sur l’entraînement.
preparations = {name: build_pipelines(quantitative)[name].named_steps["preparation"]
                for name in ["Logistique", "RandomForest"]}
matrices_train, matrices_test = {}, {}
for name, prep in preparations.items():
    matrices_train[name] = pd.DataFrame(prep.fit_transform(X_train), index=X_train.index,
                                      columns=prep.get_feature_names_out())
    matrices_test[name] = pd.DataFrame(prep.transform(X_test), index=X_test.index,
                                     columns=prep.get_feature_names_out())
travel_columns = [c for c in matrices_train["Logistique"] if c.startswith("categories_scores__BusinessTravel_")]
demo = X_train[["BusinessTravel"]].drop_duplicates().index
display(X_train.loc[demo, ["BusinessTravel"]].join(matrices_train["Logistique"].loc[demo, travel_columns]))
np.testing.assert_allclose(matrices_train["Logistique"][travel_columns].sum(axis=1), 1)

,BusinessTravel,categories_scores__BusinessTravel_Non-Travel,categories_scores__BusinessTravel_Travel_Frequently,categories_scores__BusinessTravel_Travel_Rarely
1466,Travel_Rarely,0.0,0.0,1.0
541,Non-Travel,1.0,0.0,0.0
501,Travel_Frequently,0.0,1.0,0.0


## 4. Scores : un nombre n’impose pas la même représentation
Pour la **logistique**, les neuf scores/niveaux sont aussi en one-hot : le modèle apprend une contribution par code sans imposer qu’un point ait le même effet partout. Il peut donc représenter une relation non monotone entre score et départ.

Pour **Random Forest**, on conserve les scores numériques : les arbres peuvent chercher un seuil, par exemple code ≤ 2. Cela utilise leur ordre, sans imposer une relation linéaire ni des intervalles égaux entre codes.

Les codes hors échelle restent présents conformément au choix de cette expérience. Le one-hot les isole en modalités ; la forêt utilise aussi leur position numérique. **Aucune de ces représentations ne garantit que ces codes soient valides.**

In [5]:
col = "WorkLifeBalance"
score_columns = [c for c in matrices_train["Logistique"] if c.startswith(f"categories_scores__{col}_")]
examples = X_train[[col]].drop_duplicates().sort_values(col).index
print("Logistique : une colonne par code observé dans l’entraînement")
display(X_train.loc[examples, [col]].join(matrices_train["Logistique"].loc[examples, score_columns]))
print("Random Forest : le nombre est conservé")
display(X_train.loc[examples, [col]].join(matrices_train["RandomForest"].loc[examples, [f"scores__{col}"]]))

Logistique : une colonne par code observé dans l’entraînement


,WorkLifeBalance,categories_scores__WorkLifeBalance_0,categories_scores__WorkLifeBalance_1,categories_scores__WorkLifeBalance_2,categories_scores__WorkLifeBalance_3,categories_scores__WorkLifeBalance_4,categories_scores__WorkLifeBalance_5,categories_scores__WorkLifeBalance_6
1485,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
613,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0
975,2,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1466,3,0.0,0.0,0.0,1.0,0.0,0.0,0.0
964,4,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1479,5,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1480,6,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Random Forest : le nombre est conservé


,WorkLifeBalance,scores__WorkLifeBalance
1485,0,0.0
613,1,1.0
975,2,2.0
1466,3,3.0
964,4,4.0
1479,5,5.0
1480,6,6.0


## 5. Pourquoi standardiser les quantitatives pour la logistique ?
Le revenu, l’âge et la distance n’ont pas les mêmes unités ni amplitudes. **StandardScaler** calcule `z = (valeur − moyenne_entraînement) / écart_type_entraînement`.

Une valeur de z=1 signifie un écart-type au-dessus de la moyenne d’entraînement. Les valeurs ne sont pas limitées à [0,1] et une valeur négative ne signifie pas une donnée erronée. La standardisation rend les unités plus comparables pour la pénalisation L2 ; elle ne rend pas automatiquement les distributions normales.

On ne standardise ni les binaires ni les indicatrices one-hot. Pour la forêt, les quantitatives restent dans leurs unités d’origine : ses séparations par seuil n’exigent pas une échelle commune.

In [6]:
scaler = preparations["Logistique"].named_transformers_["quantitatives"]
display(pd.DataFrame({"variable": quantitative, "moyenne_train": scaler.mean_,
                      "ecart_type_train": scaler.scale_}))
col = "MonthlyIncome"
preview = pd.DataFrame({"revenu_avant": X_train[col].head(6),
    "logistique_apres": matrices_train["Logistique"][f"quantitatives__{col}"].head(6),
    "foret_apres": matrices_train["RandomForest"][f"quantitatives__{col}"].head(6)})
display(preview)
np.testing.assert_allclose(scaler.mean_, X_train[quantitative].mean())

,variable,moyenne_train,ecart_type_train
0,Age,37.168040,9.273251
1,DailyRate,801.744646,402.988103
2,DistanceFromHome,9.162273,8.134224
3,HourlyRate,65.574135,20.412041
4,MonthlyIncome,6492.019769,4640.074884
5,MonthlyRate,14248.393740,7289.371564
6,NumCompaniesWorked,2.696046,2.489656
7,PercentSalaryHike,15.180395,3.649546
8,TotalWorkingYears,11.323723,7.728175
9,TrainingTimesLastYear,2.770181,1.287016


,revenu_avant,logistique_apres,foret_apres
1466,9991,0.754078,9991.0
964,6125,-0.079098,6125.0
975,13695,1.552341,13695.0
1280,4071,-0.521763,4071.0
662,2044,-0.958609,2044.0
541,11713,1.125193,11713.0


## 6. Que faire d’une modalité inconnue au moment de prédire ?
`OneHotEncoder(handle_unknown="ignore")` conserve les colonnes apprises à l’entraînement. Si une nouvelle catégorie apparaît, toutes les indicatrices de cette variable valent zéro. Cela évite une erreur, mais le modèle n’a pas appris de contribution spécifique à cette catégorie : il faut la surveiller.

La cellule suivante utilise **un libellé fictif uniquement pour la démonstration**, dans une copie d’une observation d’entraînement. Il n’est ni ajouté au fichier ni appris par l’encodeur. Ce comportement ne s’applique pas à nos binaires : un code binaire inattendu reste une erreur.

In [7]:
fiction = X_train.iloc[:1].copy()
fiction["JobRole"] = "FICTIF_POUR_DEMONSTRATION"
prep = preparations["Logistique"]
encoded = pd.DataFrame(prep.transform(fiction), columns=prep.get_feature_names_out())
job_columns = [c for c in encoded if c.startswith("categories_scores__JobRole_")]
display(encoded[job_columns])
assert encoded[job_columns].to_numpy().sum() == 0
assert "FICTIF_POUR_DEMONSTRATION" not in X.JobRole.unique()

,categories_scores__JobRole_Healthcare Representative,categories_scores__JobRole_Human Resources,categories_scores__JobRole_Laboratory Technician,categories_scores__JobRole_Manager,categories_scores__JobRole_Manufacturing Director,categories_scores__JobRole_Research Director,categories_scores__JobRole_Research Scientist,categories_scores__JobRole_Sales Executive,categories_scores__JobRole_Sales Representative
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7. Ce qui change, et ce qui ne change pas
Les matrices deviennent numériques et plus larges ; l’ordre des lignes reste identique. La cible reste séparée. Il n’y a pas d’apprentissage d’un classifieur dans ce notebook.

| Famille | Logistique | Random Forest |
|---|---|---|
| Binaires | 0/1 | 0/1 |
| Nominales | One-hot | One-hot |
| Scores / niveaux | One-hot | Numériques inchangés |
| Quantitatives | StandardScaler | Unités initiales |

La liste des colonnes transformées ci-dessous est celle apprise sur l’entraînement. Une modalité rare absente d’un pli peut produire moins de colonnes dans ce pli, ce qui est normal.

In [8]:
for name in preparations:
    print(name, "— avant :", X_train.shape, "— après :", matrices_train[name].shape)
    display(pd.DataFrame({"colonne_transformee": matrices_train[name].columns}))
    assert matrices_train[name].index.equals(X_train.index)
    assert matrices_test[name].index.equals(X_test.index)
    assert matrices_train[name].columns.equals(matrices_test[name].columns)
    assert np.isfinite(matrices_train[name].to_numpy()).all()

Logistique — avant : (1214, 30) — après : (1214, 93)


,colonne_transformee
0,binaires__Gender
1,binaires__OverTime
2,categories_scores__BusinessTravel_Non-Travel
3,categories_scores__BusinessTravel_Travel_Frequ...
4,categories_scores__BusinessTravel_Travel_Rarely
...,...
88,quantitatives__TrainingTimesLastYear
89,quantitatives__YearsAtCompany
90,quantitatives__YearsInCurrentRole
91,quantitatives__YearsSinceLastPromotion


RandomForest — avant : (1214, 30) — après : (1214, 49)


,colonne_transformee
0,binaires__Gender
1,binaires__OverTime
2,categories__BusinessTravel_Non-Travel
3,categories__BusinessTravel_Travel_Frequently
4,categories__BusinessTravel_Travel_Rarely
5,categories__Department_Human Resources
6,categories__Department_Research & Development
7,categories__Department_Sales
8,categories__EducationField_Human Resources
9,categories__EducationField_Life Sciences


## 8. Pourquoi un Pipeline est indispensable à la validation
Le découpage doit précéder l’ajustement des transformations. Dans une validation croisée, chaque pli possède son encodeur et son scaler ajustés uniquement sur ses lignes d’apprentissage. Dans 04 et 05, GridSearchCV reçoit **le pipeline entier**, et non une matrice déjà encodée sur tout l’entraînement.

Ici les prétraitements sont ajustés sur tout l’entraînement pour les illustrer, puis appliqués au test sans réajustement. **Ne pas réutiliser ces matrices précalculées pour faire la validation croisée** : cela laisserait les lignes de validation influencer le prétraitement.

Les variables candidates du notebook 01 ne sont pas intégrées ici. En particulier, une médiane salariale par métier/niveau devra elle aussi être apprise à l’intérieur de chaque pli si elle entre dans un modèle.

In [9]:
# Démonstration concrète : nouveau scaler appris seulement sur le premier pli.
from sklearn.base import clone
fit_indices, validation_indices = folds[0]
fold_prep = clone(preparations["Logistique"])
fold_prep.fit(X_train.iloc[fit_indices])
fold_scaler = fold_prep.named_transformers_["quantitatives"]
assert fold_scaler.n_samples_seen_ == len(fit_indices)
np.testing.assert_allclose(fold_scaler.mean_, X_train.iloc[fit_indices][quantitative].mean())
print("Pli 0 :", len(fit_indices), "lignes pour apprendre les transformations ;",
      len(validation_indices), "lignes réservées à la validation.")
pd.testing.assert_frame_equal(X, X_before)
assert sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash
print("Données et modèles sauvegardés inchangés. Aucun export créé.")

Pli 0 : 971 lignes pour apprendre les transformations ; 243 lignes réservées à la validation.
Données et modèles sauvegardés inchangés. Aucun export créé.
